# Retail Sales Data Analysis — UCI Online Retail

**Domain:** Retail / E-commerce  
**Project type:** End-to-end exploratory data analysis (EDA) and business insights  
**Author:** Rohit Labana

## Objective
Analyze real transactional retail data to understand revenue trends, product performance, customer behavior, cancellations/returns, and geographic demand.

## Dataset
The project uses the **UCI Online Retail** dataset, a real transaction dataset from a UK-based registered non-store online retailer. It covers transactions from **1 December 2010 to 9 December 2011** and contains **541,909 records** with fields including invoice number, product, quantity, invoice date, unit price, customer ID, and country.

Source: UCI Machine Learning Repository — Online Retail, DOI: 10.24432/C5BW33.

> The notebook downloads the dataset with `ucimlrepo`, so the repository does not need to store the 22.6 MB source workbook.


In [ ]:
# Install once if needed:
# !pip install ucimlrepo pandas numpy matplotlib seaborn openpyxl

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

print('Libraries loaded successfully.')

## 1. Load the real dataset

In [ ]:
online_retail = fetch_ucirepo(id=352)
df = online_retail.data.features.copy()

print('Shape:', df.shape)
display(df.head())
display(df.info())

## 2. Understand the data

In [ ]:
print('Columns:')
print(df.columns.tolist())

print('\nData types:')
display(df.dtypes)

print('\nSummary statistics:')
display(df.describe(include='all').T)

## 3. Data quality checks

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
print('Missing values:')
display(missing[missing > 0])

print('Duplicate rows:', df.duplicated().sum())
print('Cancelled invoices:', df['InvoiceNo'].astype(str).str.startswith('C').sum())
print('Negative quantity rows:', (df['Quantity'] < 0).sum())

## 4. Data cleaning and feature engineering

In [ ]:
df = df.drop_duplicates().copy()
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['InvoiceNo'] = df['InvoiceNo'].astype(str)
df['Description'] = df['Description'].astype('string')

# Cancellation/return flag
df['IsCancelled'] = df['InvoiceNo'].str.startswith('C') | (df['Quantity'] < 0)

# Revenue at line level
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# Positive-sales dataset for standard sales analysis
sales = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].copy()
sales['YearMonth'] = sales['InvoiceDate'].dt.to_period('M').astype(str)
sales['Month'] = sales['InvoiceDate'].dt.month
sales['DayOfWeek'] = sales['InvoiceDate'].dt.day_name()
sales['Hour'] = sales['InvoiceDate'].dt.hour

print('Cleaned rows:', len(df))
print('Positive-sales rows:', len(sales))
print('Total positive revenue: £{:,.2f}'.format(sales['Revenue'].sum()))

## 5. Revenue and sales trend

In [ ]:
monthly_sales = sales.groupby('YearMonth', as_index=False)['Revenue'].sum()

plt.figure(figsize=(12,5))
sns.lineplot(data=monthly_sales, x='YearMonth', y='Revenue', marker='o')
plt.title('Monthly Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Revenue (£)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('Best month:', monthly_sales.loc[monthly_sales['Revenue'].idxmax()].to_dict())

## 6. Top products

In [ ]:
top_products = (sales.groupby('Description', as_index=False)['Revenue']
                .sum()
                .sort_values('Revenue', ascending=False)
                .head(10))

display(top_products)

plt.figure(figsize=(10,6))
sns.barplot(data=top_products, y='Description', x='Revenue')
plt.title('Top 10 Products by Revenue')
plt.xlabel('Revenue (£)')
plt.ylabel('Product')
plt.tight_layout()
plt.show()

## 7. Country-level performance

In [ ]:
country_sales = (sales.groupby('Country', as_index=False)
                .agg(Revenue=('Revenue','sum'), Orders=('InvoiceNo','nunique'))
                .sort_values('Revenue', ascending=False))

display(country_sales.head(10))

plt.figure(figsize=(10,6))
sns.barplot(data=country_sales.head(10), y='Country', x='Revenue')
plt.title('Top 10 Countries by Revenue')
plt.xlabel('Revenue (£)')
plt.ylabel('Country')
plt.tight_layout()
plt.show()

## 8. Customer analysis

In [ ]:
customer_sales = (sales.dropna(subset=['CustomerID'])
                 .groupby('CustomerID', as_index=False)
                 .agg(Revenue=('Revenue','sum'),
                      Orders=('InvoiceNo','nunique'),
                      Items=('Quantity','sum'))
                 .sort_values('Revenue', ascending=False))

print('Unique identified customers:', customer_sales['CustomerID'].nunique())
display(customer_sales.head(10))

plt.figure(figsize=(8,5))
sns.histplot(customer_sales['Revenue'], bins=40, kde=True)
plt.title('Distribution of Customer Revenue')
plt.xlabel('Customer Revenue (£)')
plt.tight_layout()
plt.show()

## 9. Cancellations / returns

In [ ]:
returns = df[df['IsCancelled']].copy()

return_summary = {
    'Cancelled/return rows': len(returns),
    'Return/negative quantity units': abs(returns.loc[returns['Quantity'] < 0, 'Quantity'].sum()),
    'Negative revenue': returns.loc[returns['Revenue'] < 0, 'Revenue'].sum(),
}
print(return_summary)

return_products = (returns.groupby('Description', as_index=False)['Quantity']
                   .sum()
                   .sort_values('Quantity')
                   .head(10))
display(return_products)

## 10. Time-of-day and weekday analysis

In [ ]:
weekday_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
weekday_sales = sales.groupby('DayOfWeek')['Revenue'].sum().reindex(weekday_order)

plt.figure(figsize=(10,5))
weekday_sales.plot(kind='bar')
plt.title('Revenue by Day of Week')
plt.xlabel('Day')
plt.ylabel('Revenue (£)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('Peak sales hour:', sales.groupby('Hour')['Revenue'].sum().idxmax())

## 11. Correlation analysis

In [ ]:
corr = sales[['Quantity','UnitPrice','Revenue']].corr()
plt.figure(figsize=(6,4))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Numeric Feature Correlation')
plt.tight_layout()
plt.show()

## 12. Key business questions

In [ ]:
print('--- BUSINESS SUMMARY ---')
print(f'Positive sales revenue: £{sales["Revenue"].sum():,.2f}')
print(f'Unique products: {sales["StockCode"].nunique():,}')
print(f'Unique invoices/orders: {sales["InvoiceNo"].nunique():,}')
print(f'Unique countries: {sales["Country"].nunique():,}')
print(f'Identified customers: {sales["CustomerID"].nunique():,}')
print(f'Average order value: £{sales.groupby("InvoiceNo")["Revenue"].sum().mean():,.2f}')
print(f'Largest revenue product: {top_products.iloc[0]["Description"]}')
print(f'Top revenue country: {country_sales.iloc[0]["Country"]}')

## 13. Conclusions

This project demonstrates an end-to-end retail analytics workflow:

1. Load a real transactional dataset.
2. Inspect schema, missing values, duplicates, and cancellation records.
3. Clean the data and engineer revenue/time features.
4. Analyze revenue trends, products, countries, customers, and returns.
5. Use visualizations to communicate patterns.
6. Convert the analysis into business-oriented findings.

### Practical business uses
- Identify high-performing products and markets.
- Monitor monthly revenue changes.
- Detect cancellation/return-heavy products.
- Identify high-value customers for retention campaigns.
- Understand peak shopping periods.

### Limitations
- The dataset represents one retailer and a historical period (2010–2011).
- Revenue is calculated from quantity × unit price and should not be interpreted as accounting profit.
- Missing CustomerID values limit customer-level analysis for anonymous transactions.

**Next step:** The same cleaned features can be extended into RFM customer segmentation, cohort analysis, or a Power BI dashboard.
